This notebook applies the production valuation models to 2026/27 player data and prepares end-of-season market-value projections and supporting fields for the application.

### 1. Imports and Paths

In [1]:
import json
import joblib
import numpy as np
import pandas as pd


DATA_PATH = "../data/processed/current_season_inference_2026_27.csv"

METADATA_PATH = "../models/production_metadata.json"

MODEL_WITHOUT_HISTORY_PATH = (
    "../models/xgb_without_history_production.joblib"
)

MODEL_WITH_HISTORY_PATH = (
    "../models/xgb_with_history_production.joblib"
)

OUTPUT_PATH = (
    "../data/processed/current_season_predictions_2026_27.csv"
)

### 2. Load Inference Data and Production Artifacts

In [2]:
df = pd.read_csv(DATA_PATH)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

model_without_history = joblib.load(
    MODEL_WITHOUT_HISTORY_PATH
)

model_with_history = joblib.load(
    MODEL_WITH_HISTORY_PATH
)

FEATURES_WITHOUT_HISTORY = (
    metadata["features_without_history"]
)

FEATURES_WITH_HISTORY = (
    metadata["features_with_history"]
)

print("Rows:", len(df))
print("Model version:", metadata["model_version"])
print("Without-history features:", len(FEATURES_WITHOUT_HISTORY))
print("With-history features:", len(FEATURES_WITH_HISTORY))

Rows: 2070
Model version: v1
Without-history features: 29
With-history features: 31


### 3. Validate Inference Dataset

In [3]:
required_features = list(
    dict.fromkeys(
        FEATURES_WITHOUT_HISTORY
        + FEATURES_WITH_HISTORY
    )
)

missing_features = [
    feature
    for feature in required_features
    if feature not in df.columns
]

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

assert df["player_code_fb"].is_unique, (
    "Duplicate players found."
)

assert df["competition_id"].notna().all(), (
    "Missing competition_id found."
)

print("Input validation passed.")

Input validation passed.


### 4. Production Prediction Function

In [4]:
def predict_market_value(input_df):
    """
    Route each player to the correct production model
    and return predicted market value in euros.
    """

    data = input_df.copy()

    # Previous market value available -> history model
    history_mask = (
        data["previous_market_value_in_eur"].notna()
    )

    log_predictions = pd.Series(
        index=data.index,
        dtype=float
    )

    # With-history model
    if history_mask.any():

        X_history = data.loc[
            history_mask,
            FEATURES_WITH_HISTORY
        ]

        log_predictions.loc[history_mask] = (
            model_with_history.predict(X_history)
        )

    # Without-history model
    no_history_mask = ~history_mask

    if no_history_mask.any():

        X_no_history = data.loc[
            no_history_mask,
            FEATURES_WITHOUT_HISTORY
        ]

        log_predictions.loc[no_history_mask] = (
            model_without_history.predict(X_no_history)
        )

    # Inverse target transformation
    predictions = np.expm1(log_predictions)

    return predictions.clip(lower=0)

### 5. Baseline Predictions

In [5]:
df["model_route"] = np.where(
    df["previous_market_value_in_eur"].notna(),
    "with_history",
    "without_history"
)

df["baseline_projected_market_value_in_eur"] = (
    predict_market_value(df)
    .round()
    .astype("Int64")
)

print(df["model_route"].value_counts())

display(
    df[
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "model_route",
        ]
    ]
    .sort_values(
        "baseline_projected_market_value_in_eur",
        ascending=False
    )
    .head(20)
)

model_route
with_history       1726
without_history     344
Name: count, dtype: int64


,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,model_route
30,Lamine Yamal,Barcelona,200000000.0,211662910,with_history
153,Erling Haaland,Manchester City,200000000.0,177180508,with_history
776,Michael Olise,Bayern Munich,150000000.0,162932391,with_history
13,Kylian Mbappé,Real Madrid,180000000.0,152797713,with_history
91,Pedri,Barcelona,150000000.0,150616567,with_history
394,Declan Rice,Arsenal,120000000.0,143996738,with_history
41,Vinicius Júnior,Real Madrid,140000000.0,136977696,with_history
71,Jude Bellingham,Real Madrid,130000000.0,127748624,with_history
376,Bukayo Saka,Arsenal,110000000.0,122319214,with_history
689,João Neves,Paris SG,140000000.0,121627328,with_history


### 6. Baseline Value Change

In [6]:
df["baseline_value_change_in_eur"] = (
    df["baseline_projected_market_value_in_eur"]
    - df["previous_market_value_in_eur"]
)

df["baseline_value_change_pct"] = (
    df["baseline_value_change_in_eur"]
    / df["previous_market_value_in_eur"]
    * 100
).round(1)

df.loc[
    df["previous_market_value_in_eur"].isna(),
    [
        "baseline_value_change_in_eur",
        "baseline_value_change_pct",
    ]
] = np.nan

### 7. Scenario feature builder

In [7]:
def build_scenario(
    player_row,
    projected_games=None,
    projected_minutes=None,
    projected_goals=None,
    projected_assists=None,
    projected_yellow_cards=None,
    projected_red_cards=None,
):
    """
    Create a modified end-of-season scenario for one player.

    Only values explicitly supplied by the user are changed.
    Dependent model features are recalculated automatically.
    """

    row = player_row.copy()

    # ----------------------------------------
    # Override projected player totals
    # ----------------------------------------

    updates = {
        "projected_games": projected_games,
        "projected_minutes": projected_minutes,
        "projected_goals": projected_goals,
        "projected_assists": projected_assists,
        "projected_yellow_cards": projected_yellow_cards,
        "projected_red_cards": projected_red_cards,
    }

    for column, value in updates.items():
        if value is not None:
            row[column] = value

    # ----------------------------------------
    # Basic validation
    # ----------------------------------------

    numeric_scenario_cols = [
        "projected_games",
        "projected_minutes",
        "projected_goals",
        "projected_assists",
        "projected_yellow_cards",
        "projected_red_cards",
    ]

    for col in numeric_scenario_cols:
        if pd.notna(row[col]) and row[col] < 0:
            raise ValueError(
                f"{col} cannot be negative."
            )

    if (
        pd.notna(row["projected_games"])
        and pd.notna(row["full_season_games"])
        and row["projected_games"] > row["full_season_games"]
    ):
        raise ValueError(
            "Projected games cannot exceed full-season league games."
        )

    if (
        pd.notna(row["projected_minutes"])
        and pd.notna(row["projected_games"])
        and row["projected_minutes"]
        > row["projected_games"] * 90
    ):
        raise ValueError(
            "Projected minutes cannot exceed projected games × 90."
        )

    # ----------------------------------------
    # Map scenario to model feature names
    # ----------------------------------------

    row["total_games_played_season"] = (
        row["projected_games"]
    )

    row["total_minutes_played"] = (
        row["projected_minutes"]
    )

    row["total_goals"] = (
        row["projected_goals"]
    )

    row["total_assists"] = (
        row["projected_assists"]
    )

    row["total_yellow_cards"] = (
        row["projected_yellow_cards"]
    )

    row["total_red_cards"] = (
        row["projected_red_cards"]
    )

    # ----------------------------------------
    # Keep clean sheets consistent with games
    # ----------------------------------------

    if (
        pd.notna(row["clean_sheet_rate"])
        and pd.notna(row["projected_games"])
    ):
        row["total_clean_sheets"] = round(
            row["clean_sheet_rate"]
            * row["projected_games"]
        )

    # ----------------------------------------
    # Recalculate dependent /90 features
    # ----------------------------------------

    minutes = row["total_minutes_played"]

    if pd.isna(minutes) or minutes < 900:

        row["goals_per_90_min"] = np.nan
        row["assists_per_90_min"] = np.nan
        row["goal_contributions_per_90_min"] = np.nan
        row["yellow_cards_per_90_min"] = np.nan
        row["red_cards_per_90_min"] = np.nan

    else:

        row["goals_per_90_min"] = (
            row["total_goals"]
            * 90
            / minutes
        )

        row["assists_per_90_min"] = (
            row["total_assists"]
            * 90
            / minutes
        )

        row["goal_contributions_per_90_min"] = (
            (
                row["total_goals"]
                + row["total_assists"]
            )
            * 90
            / minutes
        )

        row["yellow_cards_per_90_min"] = (
            row["total_yellow_cards"]
            * 90
            / minutes
        )

        row["red_cards_per_90_min"] = (
            row["total_red_cards"]
            * 90
            / minutes
        )

    return row

### 8. Scenario prediction function 

In [8]:
def predict_player_scenario(
    player_name,
    projected_games=None,
    projected_minutes=None,
    projected_goals=None,
    projected_assists=None,
    projected_yellow_cards=None,
    projected_red_cards=None,
):
    """
    Build and predict one custom player scenario.
    """

    matches = df[
        df["player"]
        .str.casefold()
        .eq(player_name.casefold())
    ]

    if matches.empty:
        raise ValueError(
            f"Player '{player_name}' not found."
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple players found for '{player_name}'. "
            "Use player_id in the app."
        )

    player = matches.iloc[0].copy()

    scenario = build_scenario(
        player_row=player,
        projected_games=projected_games,
        projected_minutes=projected_minutes,
        projected_goals=projected_goals,
        projected_assists=projected_assists,
        projected_yellow_cards=projected_yellow_cards,
        projected_red_cards=projected_red_cards,
    )

    scenario_df = pd.DataFrame([scenario])

    scenario_value = (
        predict_market_value(scenario_df)
        .iloc[0]
    )

    result = {
        "player": player["player"],
        "team": player["team"],
        "model_route": (
            "with_history"
            if pd.notna(
                player["previous_market_value_in_eur"]
            )
            else "without_history"
        ),
        "previous_market_value_in_eur": (
            player["previous_market_value_in_eur"]
        ),
        "baseline_projected_market_value_in_eur": (
            player[
                "baseline_projected_market_value_in_eur"
            ]
        ),
        "scenario_projected_market_value_in_eur": round(
            scenario_value
        ),
        "projected_games": scenario["projected_games"],
        "projected_minutes": scenario["projected_minutes"],
        "projected_goals": scenario["projected_goals"],
        "projected_assists": scenario["projected_assists"],
        "projected_yellow_cards": scenario[
            "projected_yellow_cards"
        ],
        "projected_red_cards": scenario[
            "projected_red_cards"
        ],
    }

    return result, scenario_df

### 9. Test a manual scenario

In [9]:
result, scenario = predict_player_scenario(
    player_name="Bukayo Saka",
    projected_minutes=2900,
    projected_goals=18,
    projected_assists=12,
)

result

{'player': 'Bukayo Saka',
 'team': 'Arsenal',
 'model_route': 'with_history',
 'previous_market_value_in_eur': np.float64(110000000.0),
 'baseline_projected_market_value_in_eur': np.int64(122319214),
 'scenario_projected_market_value_in_eur': 122919108,
 'projected_games': np.float64(38.0),
 'projected_minutes': 2900,
 'projected_goals': 18,
 'projected_assists': 12,
 'projected_yellow_cards': np.float64(0.0),
 'projected_red_cards': np.float64(0.0)}

In [10]:
print("Player:", result["player"])
print("Team:", result["team"])
print()

print(
    "Previous value:",
    f'€{result["previous_market_value_in_eur"]:,.0f}'
)

print(
    "Baseline projection:",
    f'€{result["baseline_projected_market_value_in_eur"]:,.0f}'
)

print(
    "Manual scenario:",
    f'€{result["scenario_projected_market_value_in_eur"]:,.0f}'
)

Player: Bukayo Saka
Team: Arsenal

Previous value: €110,000,000
Baseline projection: €122,319,214
Manual scenario: €122,919,108


### 10 Inspect Scenario Model Features

In [11]:
route = result["model_route"]

if route == "with_history":
    scenario_features = FEATURES_WITH_HISTORY
else:
    scenario_features = FEATURES_WITHOUT_HISTORY

display(
    scenario[
        scenario_features
    ].T
)

,376
position,Attack
sub_position,Right Winger
foot,left
competition_id,GB1
season,2026
age_at_valuation,25.0
height_in_cm,178.0
total_games_played_season,38.0
total_minutes_played,2900
total_goals,18


### 11. Player ID Version for the Future APP

In [12]:
def predict_scenario_by_player_id(
    player_id,
    projected_games=None,
    projected_minutes=None,
    projected_goals=None,
    projected_assists=None,
    projected_yellow_cards=None,
    projected_red_cards=None,
):

    matches = df[
        df["player_id"] == player_id
    ]

    if matches.empty:
        raise ValueError(
            f"player_id {player_id} not found."
        )

    player = matches.iloc[0].copy()

    scenario = build_scenario(
        player_row=player,
        projected_games=projected_games,
        projected_minutes=projected_minutes,
        projected_goals=projected_goals,
        projected_assists=projected_assists,
        projected_yellow_cards=projected_yellow_cards,
        projected_red_cards=projected_red_cards,
    )

    scenario_df = pd.DataFrame([scenario])

    predicted_value = (
        predict_market_value(scenario_df)
        .iloc[0]
    )

    return {
        "player_id": player_id,
        "player": player["player"],
        "team": player["team"],
        "previous_market_value_in_eur": (
            player["previous_market_value_in_eur"]
        ),
        "baseline_projected_market_value_in_eur": (
            player[
                "baseline_projected_market_value_in_eur"
            ]
        ),
        "scenario_projected_market_value_in_eur": (
            round(predicted_value)
        ),
    }

### 12. Baseline Diagnostics

In [13]:
display(
    df[
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "baseline_value_change_in_eur",
            "baseline_value_change_pct",
            "model_route",
        ]
    ]
    .sort_values(
        "baseline_projected_market_value_in_eur",
        ascending=False
    )
    .head(30)
)

,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,baseline_value_change_in_eur,baseline_value_change_pct,model_route
30,Lamine Yamal,Barcelona,200000000.0,211662910,11662910.0,5.8,with_history
153,Erling Haaland,Manchester City,200000000.0,177180508,-22819492.0,-11.4,with_history
776,Michael Olise,Bayern Munich,150000000.0,162932391,12932391.0,8.6,with_history
13,Kylian Mbappé,Real Madrid,180000000.0,152797713,-27202287.0,-15.1,with_history
91,Pedri,Barcelona,150000000.0,150616567,616567.0,0.4,with_history
394,Declan Rice,Arsenal,120000000.0,143996738,23996738.0,20.0,with_history
41,Vinicius Júnior,Real Madrid,140000000.0,136977696,-3022304.0,-2.2,with_history
71,Jude Bellingham,Real Madrid,130000000.0,127748624,-2251376.0,-1.7,with_history
376,Bukayo Saka,Arsenal,110000000.0,122319214,12319214.0,11.2,with_history
689,João Neves,Paris SG,140000000.0,121627328,-18372672.0,-13.1,with_history


In [14]:
display(
    df[
        df["previous_market_value_in_eur"].notna()
    ][
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "baseline_value_change_in_eur",
            "baseline_value_change_pct",
        ]
    ]
    .sort_values(
        "baseline_value_change_in_eur",
        ascending=False
    )
    .head(20)
)

,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,baseline_value_change_in_eur,baseline_value_change_pct
309,Luka Vušković,Brighton,60000000.0,86775535,26775535.0,44.6
394,Declan Rice,Arsenal,120000000.0,143996738,23996738.0,20.0
480,Franco Mastantuono,Fiorentina,45000000.0,65700808,20700808.0,46.0
830,Yan Diomandé,Real Madrid,90000000.0,109450869,19450869.0,21.6
790,Jobe Bellingham,Dortmund,40000000.0,57349875,17349875.0,43.4
754,Aleksandar Pavlovic,Bayern Munich,90000000.0,107137747,17137747.0,19.0
382,Jérémy Jacquet,Liverpool,55000000.0,70753917,15753917.0,28.6
373,Riccardo Calafiori,Arsenal,55000000.0,69816328,14816328.0,26.9
392,Alexander Isak,Liverpool,85000000.0,99704061,14704061.0,17.3
362,Pau Cubarsí,Barcelona,80000000.0,94071144,14071144.0,17.6


In [15]:
display(
    df[
        df["previous_market_value_in_eur"].notna()
    ][
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "baseline_value_change_in_eur",
            "baseline_value_change_pct",
        ]
    ]
    .sort_values(
        "baseline_value_change_in_eur"
    )
    .head(20)
)

,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,baseline_value_change_in_eur,baseline_value_change_pct
13,Kylian Mbappé,Real Madrid,180000000.0,152797713,-27202287.0,-15.1
1114,Ousmane Dembélé,Paris SG,100000000.0,75166178,-24833822.0,-24.8
153,Erling Haaland,Manchester City,200000000.0,177180508,-22819492.0,-11.4
847,Khvicha Kvaratskhelia,Paris SG,140000000.0,117612598,-22387402.0,-16.0
1583,Julián Álvarez,Atlético Madrid,100000000.0,78925376,-21074624.0,-21.1
785,Désiré Doué,Paris SG,120000000.0,101263355,-18736645.0,-15.6
689,João Neves,Paris SG,140000000.0,121627328,-18372672.0,-13.1
343,Sandro Tonali,Tottenham,80000000.0,65453653,-14546347.0,-18.2
1949,Moisés Caicedo,Chelsea,100000000.0,87067822,-12932178.0,-12.9
1211,Nuno Mendes,Paris SG,80000000.0,67981370,-12018630.0,-15.0


In [16]:
display(
    df[
        df["model_route"] == "without_history"
    ][
        [
            "player",
            "team",
            "age_at_valuation",
            "baseline_projected_market_value_in_eur",
        ]
    ]
    .sort_values(
        "baseline_projected_market_value_in_eur",
        ascending=False
    )
    .head(30)
)

,player,team,age_at_valuation,baseline_projected_market_value_in_eur
580,João Pedro,Chelsea,25.0,66659116
816,Francesco Esposito,Inter,21.0,65782564
688,Wesley,Roma,23.0,45859208
404,Matheus Cunha,Manchester Utd,28.0,43135992
680,Kouakou Gadou,Dortmund,20.0,42781452
370,Gabriele Bracaglia,Frosinone,23.0,42747194
329,Diego Gómez,Brighton,24.0,41883944
708,Emanuele Lulli,Roma,19.0,40611884
299,Flávio Nazinho,Monaco,23.0,37812945
645,Felix Keidel,Elversberg,23.0,37272606


### 13. Save Baseline Production Dataset

In [17]:
df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Saved: ../data/processed/current_season_predictions_2026_27.csv
Rows: 2070
Columns: 58


### 14. Final Validation

In [18]:
assert df["player_code_fb"].is_unique

assert (
    df["baseline_projected_market_value_in_eur"]
    .notna()
    .all()
)

assert (
    df["baseline_projected_market_value_in_eur"] >= 0
).all()

assert set(
    df["model_route"].unique()
).issubset(
    {
        "with_history",
        "without_history",
    }
)

print("Production prediction engine ready.")

Production prediction engine ready.


In [19]:
# 1. Extreme predictions
display(
    df[
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "model_route",
        ]
    ]
    .sort_values("baseline_projected_market_value_in_eur", ascending=False)
    .head(10)
)

display(
    df[
        [
            "player",
            "team",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
            "model_route",
        ]
    ]
    .sort_values("baseline_projected_market_value_in_eur")
    .head(10)
)

,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,model_route
30,Lamine Yamal,Barcelona,200000000.0,211662910,with_history
153,Erling Haaland,Manchester City,200000000.0,177180508,with_history
776,Michael Olise,Bayern Munich,150000000.0,162932391,with_history
13,Kylian Mbappé,Real Madrid,180000000.0,152797713,with_history
91,Pedri,Barcelona,150000000.0,150616567,with_history
394,Declan Rice,Arsenal,120000000.0,143996738,with_history
41,Vinicius Júnior,Real Madrid,140000000.0,136977696,with_history
71,Jude Bellingham,Real Madrid,130000000.0,127748624,with_history
376,Bukayo Saka,Arsenal,110000000.0,122319214,with_history
689,João Neves,Paris SG,140000000.0,121627328,with_history


,player,team,previous_market_value_in_eur,baseline_projected_market_value_in_eur,model_route
1837,Josan,Elche,100000.0,155944,with_history
2027,Mathis Saka,Toulouse,150000.0,172791,with_history
5,Matías Dituro,Elche,100000.0,189308,with_history
1292,Vincent Sasso,Le Havre,200000.0,194709,with_history
1990,Berkay Yılmaz,Freiburg,25000.0,212783,with_history
2008,Joselu Pérez,Getafe,100000.0,247461,with_history
1154,Samuel Yohou,Le Mans,250000.0,253063,with_history
2034,Yaya Diémé,Strasbourg,300000.0,257729,with_history
1896,Ismaïl Bouneb,Le Havre,100000.0,268892,with_history
1169,Pedro Bigas,Elche,300000.0,277402,with_history


In [20]:
# 2. Routing check
assert (
    df.loc[
        df["previous_market_value_in_eur"].notna(),
        "model_route"
    ] == "with_history"
).all()

assert (
    df.loc[
        df["previous_market_value_in_eur"].isna(),
        "model_route"
    ] == "without_history"
).all()

print("Routing OK")

Routing OK


In [21]:
# 3. Basic feature sanity
display(
    df[
        [
            "player",
            "projected_minutes",
            "projected_goals",
            "projected_assists",
            "age_at_valuation",
            "previous_market_value_in_eur",
            "baseline_projected_market_value_in_eur",
        ]
    ]
    .sample(10, random_state=42)
)

,player,projected_minutes,projected_goals,projected_assists,age_at_valuation,previous_market_value_in_eur,baseline_projected_market_value_in_eur
1181,Jonathan Tah,1589.500000,0.000000,0.000000,31.0,28000000.0,26826335
69,Abel Bretones,2877.142857,0.000000,0.000000,26.0,3500000.0,4559125
351,Jonathan Clauss,2890.000000,0.000000,0.000000,34.0,4000000.0,3042418
1163,Patrick Cutrone,1466.800000,0.000000,0.000000,29.0,5000000.0,3569704
429,Luismi,2149.714286,5.428571,5.428571,26.0,NaN,9004586
1930,Reigan Heskey,136.000000,0.000000,0.000000,19.0,NaN,997172
239,Charlie Cresswell,3060.000000,0.000000,0.000000,24.0,25000000.0,29067438
1749,Senne Lynen,382.500000,0.000000,0.000000,28.0,8000000.0,6536609
855,Lucas Calodat,1883.600000,6.800000,6.800000,25.0,100000.0,851532
464,Kenny Lala,2590.800000,0.000000,0.000000,35.0,1500000.0,1163485


In [22]:
# 4. Scenario sensitivity
player_name = "Bukayo Saka"

for goals in [5, 10, 20]:
    result, _ = predict_player_scenario(
        player_name=player_name,
        projected_goals=goals
    )

    print(
        goals,
        "goals ->",
        f'€{result["scenario_projected_market_value_in_eur"]:,.0f}'
    )

5 goals -> €115,967,768
10 goals -> €117,337,893
20 goals -> €121,455,779
